In [0]:
from scripts.utils import silver_utils
print(dir(silver_utils))

In [0]:
from pyspark.sql.functions import (
    col, when, sum, lit, trim, lower, upper,
    to_date, current_timestamp,round
)
from datetime import datetime
import uuid,logging

In [0]:
df=spark.read.table("workspace.ecommerce_bronze.df_payments")
logging.info('table read successfully')

In [0]:
silver_utils.check_schema(df)

In [0]:
df=silver_utils.cast_types(df,{
    "order_id" : "string",
    "payment_sequential" : "integer",
    "payment_type" : "string",
    "payment_installments" : "integer",
    "payment_value" : "double"
})

In [0]:
silver_utils.null_profiling(df,"df_payments")
df=silver_utils.handle_nulls_drop(df,drop_cols=["order_id","payment_value"])

In [0]:
df=silver_utils.standardize_strings(df,{
    "payment_type" : lambda c : lower(trim(c)),
    "order_id" : lambda c : trim(c)
    })

In [0]:
df_orders= spark.read.table("workspace.ecommerce_bronze.df_orders")
df_with_check = (
    df.alias("payments")
    .join(df_orders.select("order_id").alias("orders"),
          col("payments.order_id") == col("orders.order_id"),
          "left")
)
checks_ref_orders = [
    (
        "orders.order_id",
        "order_id exists in df_customers",
        col("orders.order_id").isNotNull()
    ),
]

dq_ref_orders= silver_utils.build_dq_table(
    spark=spark,
    df=df_with_check,
    checks=checks_ref_orders,
    table_name="df_payments",
    warn_threshold=5.0
)

checks = [
    #nulls
    (
        "order_id",
        "order_id not null",
        col("order_id").isNotNull()
    ),
    (
        "payment_value",
        "payment_value not null",
        col("payment_value").isNotNull()
    ),

    #business rules
    (
        "payment_sequential",
        "payment_sequential > 0",
        col("payment_sequential") > 0
    ),
    (
        "payment_value",
        "payment_value > 0",
        col("payment_value") > 0
    ),
    (
        "payment_installments",
        "payment_installments > 0",
        col("payment_installments") > 0
    ),
    (
        "payment_type",
        "payment_type in ('credit_card','wallet','voucher','debit_card')",
        col("payment_type").isin(["credit_card","wallet","voucher","debit_card"])
    )
]

dq_table = silver_utils.build_dq_table(
    spark=spark,
    df=df,
    checks=checks,
    table_name="df_payments",
    warn_threshold=5.0
)

#combine all dq results into one table
from functools import reduce
from pyspark.sql import DataFrame

dq_final = reduce(DataFrame.union, [dq_table, dq_ref_orders])

dq_final.write.mode("overwrite").saveAsTable("workspace.ecommerce_silver.df_payments_dq")
dq_saved = spark.read.table("workspace.ecommerce_silver.df_payments_dq")
display(dq_saved)

In [0]:
df=df.withColumn("is_successfull_payment",when(col("payment_value") > 0, lit(True)).otherwise(lit(False)))
df=df.withColumn("payment_value_with_tax",round(col("payment_value")* 1.14,2))

In [0]:
df =silver_utils.add_silver_metadata(df)

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.ecommerce_silver.df_payments")
print("saved successfully !")

In [0]:
%sql
--sanity check
SELECT * FROM workspace.ecommerce_silver.df_payments LIMIT 10